<a href="https://colab.research.google.com/github/csk01/lpg-cylinder-detection/blob/main/lpg_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# imports
# roboflow: pulls the source detection dataset; ultralytics: runs the YOLO detector for cropping
!pip install roboflow ultralytics -q
from google.colab import files
from google.colab import userdata
from roboflow import Roboflow


In [ ]:
#uploading best.pt
# Trained YOLOv11 detector checkpoint used to find/crop cylinders in the source images below
uploaded = files.upload()

In [ ]:
# Requires a Colab secret named ROBOFLOW_API_KEY (Colab -> Secrets) with access to this project
roboflow_api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(roboflow_api_key)
# Source detection dataset (images + bounding boxes) that crops will be generated from
project = rf.workspace("krishnans-workspace-eu0zb").project("lpg-identification-v2")
version = project.version(2)
dataset = version.download("yolov11")

In [ ]:
# Cell 4 — generate crops
from ultralytics import YOLO
model = YOLO("best.pt")
# conf=0.6: higher-than-default detection threshold to avoid low-confidence/false-positive crops
# save_crop=True writes each detected cylinder region as its own cropped image file
results = model(
    f"{dataset.location}/train/images/",
    conf=0.6,
    save_crop=True,
    project="lpg-crops",
    name="v1"
)
print("Crops done!")

In [ ]:
# Sanity check: confirm how many crops YOLO produced and preview filenames
import os
crops_dir = "/content/runs/detect/lpg-crops/v1/crops/lpg-cylinder"
files = os.listdir(crops_dir)
print(f"Total crops: {len(files)}")
print("Sample files:")
for f in files[:10]:
    print(f)

In [ ]:
from PIL import Image
import shutil

output_dir = "/content/classification_crops"
os.makedirs(output_dir, exist_ok=True)

kept = 0
skipped = 0

# Filter out crops that are too small to be useful for classifier training (>80x80 px kept)
for img_file in os.listdir(crops_dir):
    img_path = os.path.join(crops_dir, img_file)
    try:
        img = Image.open(img_path)
        w, h = img.size
        if w > 80 and h > 80:
            shutil.copy(img_path, output_dir)
            kept += 1
        else:
            skipped += 1
    except:
        skipped += 1

print(f"✅ Kept: {kept}")
print(f"❌ Skipped: {skipped}")

In [ ]:
# Pre-create brand subfolders so the downloaded zip is ready for manual sorting downstream
for brand in ["indane", "bharat_gas", "hp_gas", "unknown"]:
    os.makedirs(f"/content/classification_crops/{brand}", exist_ok=True)
print("Folders created!")

## Save output + download

Zips the filtered crops (plus the empty brand subfolders) and downloads the archive locally for
manual sorting.


In [ ]:
# Package filtered crops + empty brand folders into a single zip and trigger a browser download
import shutil
shutil.make_archive("/content/crops_to_sort", 'zip', "/content/classification_crops")

from google.colab import files
files.download("/content/crops_to_sort.zip")

## Output Files

| File | Saved to | Contents / purpose |
|---|---|---|
| `lpg-crops/v1/crops/lpg-cylinder/*.jpg` | `/content/runs/detect/lpg-crops/v1/crops/lpg-cylinder/` | Raw YOLO-detected cylinder crops before size filtering |
| `classification_crops/*.jpg` | `/content/classification_crops/` | Crops that passed the >80×80px size filter |
| `classification_crops/{indane,bharat_gas,hp_gas,unknown}/` | `/content/classification_crops/` | Empty brand subfolders created for manual sorting |
| `crops_to_sort.zip` | `/content/crops_to_sort.zip` (downloaded to local machine) | Final zipped output of this notebook — filtered crops + empty brand folders, ready for a human to manually sort into brand labels |
